In [ ]:
import pandas as pd 
import numpy as np 
from pathlib import Path
DATAPATH = Path("dataset") 
orders = pd.read_csv(DATAPATH / "orders.csv") 
proucts= pd.read_csv(DATAPATH / "products.csv") 
aisles = pd.read_csv(DATAPATH / "aisles.csv") 
products_prior = pd.read_csv(DATAPATH / "order_products__prior.csv") 
products_train = pd.read_csv(DATAPATH / "order_products__train.csv")
print("Order Shape:",orders.shape) 
print("Products Shape:",proucts.shape) 
print("Aisles Shape:",aisles.shape) 
print("Products Prior Shape:",products_prior.shape) 
print("Products Train Shape:",products_train.shape)

In [ ]:
orders.head(5) 
user_order = products_prior.merge(orders[["order_id", "user_id",'order_number']], on="order_id",how="left") 
user_order.head(5) 
user_products = user_order.groupby(["user_id", "product_id"]).size().reset_index(name="purchase_count") 
user_products.head(5)

In [ ]:
products_train = products_train.merge(orders[["order_id", "user_id"]], on="order_id", how="left") products_train['reordered']=1
train = products_train[["user_id","product_id","reordered"]] 
train.head(5)

In [ ]:
dataset=user_products.merge(train,on=["user_id","product_id"],how="left") dataset['reordered']=dataset['reordered'].fillna(0)

In [ ]:
dataset.head(5) 
print(dataset.shape)
print(dataset['reordered'].mean())

In [ ]:
user_last_order = orders.groupby("user_id")["order_number"].max().reset_index(name="max_order_number")

In [ ]:
user_last_order.head(5)

In [ ]:
user_product_last_order = (user_order.groupby(["user_id", "product_id"])["order_number"].max().reset_index(name="last_product_order"))

In [ ]:
user_product_last_order.head(5)

In [ ]:
recency = user_product_last_order.merge(user_last_order, on="user_id", how="left")

In [ ]:
recency["order_gap"] = recency["max_order_number"] - recency["last_product_order"]

In [ ]:
recency.head(5)

In [ ]:
dataset=dataset.merge(recency[["user_id", "product_id", "order_gap"]], on=["user_id", "product_id"], how="left")

In [ ]:
dataset.head(5)

In [ ]:
user_total_orders = user_last_order.copy()

In [ ]:
user_total_orders.rename(columns={"max_order_number": "user_total_orders"}, inplace=True)

In [ ]:
user_total_orders.head(5)

In [ ]:
dataset=dataset.merge(user_total_orders, on="user_id", how="left")

In [ ]:
dataset.head(5)

In [ ]:
dataset['purchase_ratio'] = dataset['purchase_count'] / dataset['user_total_orders']

In [ ]:
dataset.head(5)

In [ ]:
product_popularity = user_order.groupby("product_id").size().reset_index(name="product_popularity")

In [ ]:
product_popularity.head(5)

In [ ]:
dataset = dataset.merge(product_popularity, on="product_id", how="left")

In [ ]:
dataset.head(5)

In [ ]:
print(dataset.isnull().sum()) 
print(dataset.describe())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import roc_auc_score

In [ ]:
X= dataset.drop(columns=["user_id", "product_id", "reordered"]) 
y=dataset["reordered"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
model=LogisticRegression(max_iter=200,solver='saga',class_weight='balanced',n_jobs=-1) 
model.fit(X_train,y_train)

In [ ]:

y_pred_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
auc_score = roc_auc_score(y_test, y_pred_proba)

In [ ]:
print(f"AUC Score: {auc_score}")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier 
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
gb_model = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=5, random_state=42)

In [ ]:
gb_model.fit(X_train, y_train)

In [ ]:
y_pred_proba_gb = gb_model.predict_proba(X_test)[:, 1]

In [ ]:
auc_score_gb = roc_auc_score(y_test, y_pred_proba_gb)

In [ ]:
print(f"Gradient Boosting AUC Score: {auc_score_gb}")

In [ ]:
dataset['reorder_probability'] = gb_model.predict_proba(X)[:, 1]

In [ ]:
top_k = 5 recommendations = ( dataset .sort_values(['user_id', 'reorder_probability'], ascending=[True, False]) .groupby('user_id') .head(top_k) )

In [ ]:
recommendations.head(20)